# Use Case — Prioritizing Bus-Stop Cooling Interventions

**Who this is for**  
Urban planners, transit-authority analysts, and climate-adaptation leads who have *their own* infrastructure data (bus stops, public benches, playgrounds, bike-share docks, shelters, schools) and need to decide **which locations to treat first** when budget is limited.

**The scenario**  
Summer heat is making your city's bus stops unbearable. Ridership dips, complaints pile up, and the council has just approved a cooling-intervention budget — trees, shade structures, reflective pavement. You have a list of bus stops. You do **not** have the budget to treat all of them. You need a data-backed, defensible shortlist.

This notebook combines **your data** (a bus-stops point layer) with **FortyGuard's layers** (heatmap, satellite segmentation, street view, environmental parameters) to answer four questions in sequence:

1. **Which stops are actually hot?**  ← heatmap × your points
2. **Why are they hot?**  ← satellite segmentation on the top hotspots
3. **What does that look like on the ground?**  ← street view on the #1 stop
4. **When is heat at its worst here?**  ← environmental parameters profile

The final output is a prioritized action list — one row per stop, ranked by temperature, with a dominant cause and a recommended intervention.

> **Bring your own data.** The notebook ships with a sample CSV at `data/sample_bus_stops.csv`. Swap in your own CSV with the same columns (`stop_id`, `name`, `latitude`, `longitude`) at Step 1 and everything downstream just works.

---

## Setup

Load `.env`, instantiate the client, define the study area. Run `notebooks/00_setup.ipynb` first if this cell errors out.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

client = FortyGuardClient()

# Study parameters. Change the date/time for the hour you want to prioritize on.
AOI              = SAN_JOSE_POLYGON      # ~104 km² (~40 mi²) across central San Jose
STUDY_DATE       = '2024-07-15'
STUDY_HOUR       = '14:00'               # design-peak afternoon
GRANULARITY_M    = 100                   # 100 m → ~10 k tiles over this AOI; 80 m would be ~16 k
TOP_N_TO_DIAGNOSE = 3                    # deepen analysis on this many hottest stops

print(f'Authenticated to {client.base_url}')
print(f'Study hour: {STUDY_DATE} {STUDY_HOUR}')

---
## Step 1 — Load your data

### What you are doing
Reading a bus-stops point layer from CSV. The schema is minimal — `stop_id`, `name`, `latitude`, `longitude` — so you can export this directly from the transit agency's GIS, a GTFS feed, or a spreadsheet.

### Why this matters
Everything downstream is built around the geometry of **your** assets. By starting from your own data, the outputs land in your existing workflow: same IDs, same names, same coordinate system. That is the difference between a dashboard and something the operations team can act on.

In [ ]:
# Swap this path for your own CSV — same four columns.
stops = pd.read_csv(ROOT / 'data' / 'sample_bus_stops.csv')
print(f'Loaded {len(stops)} stops')
stops.head()

In [ ]:
# Plot the stops as a quick sanity check.
center = [stops['latitude'].mean(), stops['longitude'].mean()]
fmap = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
for _, r in stops.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=5, color='#1f77b4', fill=True, fill_opacity=0.9,
        popup=f"{r.stop_id} — {r['name']}",
    ).add_to(fmap)
folium.GeoJson(AOI, style_function=lambda _: {'color': '#555', 'fill': False, 'weight': 1, 'dashArray': '5,5'}).add_to(fmap)
fmap.fit_bounds([[stops['latitude'].min(), stops['longitude'].min()],
                 [stops['latitude'].max(), stops['longitude'].max()]])
fmap

---
## Step 2a — Create the heat layer (via API)

### What you are doing
Requesting a high-resolution heatmap over the study AOI at the design-peak hour. The response is a GeoJSON tile layer with a temperature value on every tile.

### Why this matters
Weather stations give you one number for the whole city. A heatmap gives you temperature **at the spatial resolution your decisions are made** — block by block. That is what lets you separate hot stops from merely-average stops.

> Run **either** Step 2a (live API call, consumes credits) **or** Step 2b (load a cached sample, free). Both produce the same `map_data` / `features` / `t_stats` variables, so everything downstream works identically.

In [ ]:
heatmap = client.create_heatmap(
    polygon_aoi=AOI,
    start_date=STUDY_DATE,
    start_time=STUDY_HOUR,
    filter_type=1,
    granularity=GRANULARITY_M,
)

map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []

stats = heatmap['result'].get('stats_data', {})
t_stats = stats.get('Temperature_stats') or stats.get('temperature_stats') or {}
print(f'Tiles returned       : {len(features)}')
print(f'AOI temperature stats: {t_stats}')

---
## Step 2b — Or: load a pre-generated heatmap (for testing)

### What you are doing
Loading a cached heatmap from `data/san_jose_heatmap_sample.geojson` instead of calling the API. Tile properties (`min_temperature`, `max_temperature`, `average_temperature` in °F) are normalized into a single `temperature` property in °C so the rest of the notebook sees the same shape Step 2a would produce.

### Why this matters
Use this path when iterating on the downstream logic without burning API credits on a heatmap you already have. Skip it on a real run — Step 2a gives you a heatmap for the exact date, hour, and AOI you care about.

In [ ]:
import json

HEATMAP_PATH = ROOT / 'data' / 'san_jose_heatmap_sample.geojson'
with open(HEATMAP_PATH, 'r') as f:
    map_data = json.load(f)

# Normalize each feature so it carries a single `temperature` (°C) property,
# mirroring what client.create_heatmap(...) would return.
def _f_to_c(t):
    return None if t is None else round((float(t) - 32) * 5 / 9, 2)

features = map_data.get('features', [])
for feat in features:
    props = feat.setdefault('properties', {})
    avg_f = props.get('average_temperature')
    props['temperature'] = _f_to_c(avg_f)

temps = [f['properties']['temperature'] for f in features if f['properties'].get('temperature') is not None]
t_stats = {
    'count': len(temps),
    'min': round(min(temps), 2) if temps else None,
    'max': round(max(temps), 2) if temps else None,
    'mean': round(sum(temps) / len(temps), 2) if temps else None,
}
print(f'Loaded heatmap       : {HEATMAP_PATH.name}')
print(f'Tiles returned       : {len(features)}')
print(f'AOI temperature stats: {t_stats}  (°C)')

---
## Step 3 — Correlate your data with the heat layer

### What you are doing
For each bus stop, finding the heatmap tile that contains it and copying that tile's temperature onto the stop. This is the **spatial join** — the moment where your asset layer and our thermal layer merge into one table.

### Why this matters
Before this step, "it's hot in the city" was a general observation. After this step, every row in your bus-stops table carries a specific temperature at the design hour. You can now sort, filter, group by route, report by neighborhood — anything you would normally do with your operational data.

In [ ]:
# Build shapely geometries once, then assign each stop its containing tile temperature.
tile_polys = [(shape(f['geometry']), f['properties'].get('temperature')) for f in features]

def _stop_temperature(lat: float, lon: float):
    p = Point(lon, lat)
    # Preferred: tile that contains the point.
    for poly, temp in tile_polys:
        if poly.contains(p):
            return temp
    # Fallback: nearest tile by centroid distance.
    if not tile_polys:
        return None
    nearest = min(tile_polys, key=lambda pt: pt[0].centroid.distance(p))
    return nearest[1]

stops['temperature_c'] = stops.apply(
    lambda r: _stop_temperature(r['latitude'], r['longitude']), axis=1
)
stops[['stop_id', 'name', 'temperature_c']].head()

---
## Step 4 — Rank and visualize

### What you are doing
Sorting stops by the temperature value we just attached, then plotting them on a map with the heatmap tiles as the backdrop. Marker color scales with stop temperature; hottest stops jump out visually and land at the top of the table.

### Why this matters
This is the first concrete deliverable — a ranked short-list of candidate locations. Even without the downstream diagnostic steps, this alone is more actionable than any citywide average the council has seen.

In [ ]:
ranked = stops.sort_values('temperature_c', ascending=False).reset_index(drop=True)
ranked.insert(0, 'rank', ranked.index + 1)
ranked

In [ ]:
# Map: heatmap tiles in the background, bus stops colored by temperature on top.
lo, hi = ranked['temperature_c'].min(), ranked['temperature_c'].max()

def _tile_style(feat):
    t = feat['properties'].get('temperature', lo)
    frac = 0 if hi == lo else (t - lo) / (hi - lo)
    r, b = int(255*frac), int(255*(1-frac))
    return {'fillColor': f'#{r:02x}00{b:02x}', 'color': '#00000000', 'fillOpacity': 0.35, 'weight': 0}

def _stop_color(t):
    if t is None: return '#888'
    frac = 0 if hi == lo else (t - lo) / (hi - lo)
    r, b = int(255*frac), int(255*(1-frac))
    return f'#{r:02x}00{b:02x}'

fmap = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
if features:
    folium.GeoJson(map_data, style_function=_tile_style).add_to(fmap)
for _, r in ranked.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=8, color='black', weight=1,
        fill=True, fill_color=_stop_color(r.temperature_c), fill_opacity=0.95,
        popup=f"#{r['rank']}  {r['stop_id']} — {r['name']}<br/>{r.temperature_c:.1f} °C",
    ).add_to(fmap)
fmap.fit_bounds([[ranked['latitude'].min(), ranked['longitude'].min()],
                 [ranked['latitude'].max(), ranked['longitude'].max()]])
fmap

---
## Step 5 — Zoom in on above-average hotspots

### What you are doing
Filtering both the heatmap tiles and the bus stops to the band `mean < temperature ≤ max` and redrawing the same map. The cooler half of the AOI falls away; only the genuinely-hot tiles and the stops inside them remain.

### Why this matters
When every tile is drawn, the eye gets pulled to whatever is warmest in the visible frame — which may still be near the citywide average. Filtering to above-mean sharpens the question: *of the stops that are hotter than typical for the AOI at this hour, where are they clustered?* That's the cluster map the council should see first.

In [ ]:
mean_t = t_stats['mean']
max_t  = t_stats['max']

hot_features = [
    f for f in features
    if f['properties'].get('temperature') is not None
    and f['properties']['temperature'] > mean_t
    and f['properties']['temperature'] <= max_t
]
hot_map_data = {'type': 'FeatureCollection', 'features': hot_features}

hot_stops = ranked[(ranked['temperature_c'] > mean_t) &
                   (ranked['temperature_c'] <= max_t)].copy()

print(f'AOI mean: {mean_t:.2f} °C   max: {max_t:.2f} °C')
print(f'Tiles above mean: {len(hot_features)} / {len(features)}')
print(f'Stops above mean: {len(hot_stops)} / {len(ranked)}')

fmap_hot = folium.Map(location=center, zoom_start=12, tiles='cartodbpositron')
if hot_features:
    folium.GeoJson(hot_map_data, style_function=_tile_style).add_to(fmap_hot)
for _, r in hot_stops.iterrows():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=8, color='black', weight=1,
        fill=True, fill_color=_stop_color(r.temperature_c), fill_opacity=0.95,
        popup=f"#{r['rank']}  {r['stop_id']} — {r['name']}<br/>{r.temperature_c:.1f} °C",
    ).add_to(fmap_hot)
if len(hot_stops):
    fmap_hot.fit_bounds([[hot_stops['latitude'].min(), hot_stops['longitude'].min()],
                         [hot_stops['latitude'].max(), hot_stops['longitude'].max()]])
fmap_hot

---
## Step 6 — Diagnose the top hotspots (why are they hot?)

### What you are doing
Running satellite segmentation on the top-N hottest stops. The API classifies the surroundings of each point into surface classes (rooftops, roads, vegetation, water, bare land). We collect the percentages into the same DataFrame.

### Why this matters
Knowing a stop is hot is not actionable on its own — *intervention selection depends on the cause*. Planting trees fixes a low-vegetation problem; reflective paving fixes a high-impervious problem; a shade structure fixes a high sky-exposure problem. Satellite segmentation tells you which of those is the dominant driver for each candidate stop.

In [ ]:
IMPERVIOUS_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings', 'rooftop', 'rooftops', 'bare'}
VEGETATION_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segments: dict, keys: set) -> float:
    total = 0.0
    for cls, pct in segments.items():
        if any(k in cls.lower() for k in keys):
            try:
                total += float(pct)
            except (TypeError, ValueError):
                pass
    return round(total, 1)

top = ranked.head(TOP_N_TO_DIAGNOSE).copy()
impervious, vegetation, raw_segments = [], [], []

for _, r in top.iterrows():
    print(f"Diagnosing #{r['rank']}  {r['stop_id']} — {r['name']}")
    sat = client.satellite_segmentation(
        latitude=r.latitude, longitude=r.longitude,
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=1, granularity=GRANULARITY_M,
        verbose=False,
    )
    segs = sat['result'].get('segmentation', {}).get('segments', {}) or {}
    raw_segments.append(segs)
    impervious.append(_bucket(segs, IMPERVIOUS_KEYS))
    vegetation.append(_bucket(segs, VEGETATION_KEYS))

top['impervious_pct'] = impervious
top['vegetation_pct'] = vegetation
top[['rank', 'stop_id', 'name', 'temperature_c', 'impervious_pct', 'vegetation_pct']]

---
## Step 7 — Ground-truth the #1 stop with street view

### What you are doing
Running street view segmentation at the hottest stop, oriented down the street. The API returns a ground-level image plus a pixel-wise segmentation.

### Why this matters
Satellite view shows *surroundings from above*. Street view shows *what a rider waiting at that stop actually sees*. That perspective is where you confirm whether a shade structure is feasible, whether there is room for trees, and whether the shelter itself (a metal box that absorbs heat) is part of the problem.

In [ ]:
hot1 = top.iloc[0]
street = client.street_view_segmentation(
    latitude=hot1.latitude, longitude=hot1.longitude,
    vertical_angle=5.0, horizontal_angle=0.0, back_view=False,
    verbose=False,
)
front = street['result'].get('front', {})
front_segments = front.get('segments', {}) or {}

sky_pct = _bucket(front_segments, {'sky'})
building_pct = _bucket(front_segments, {'building', 'buildings', 'wall'})
print(f"#1 stop: {hot1['stop_id']} — {hot1['name']}")
print(f"  sky fraction     : {sky_pct}%  (→ higher = more shade needed)")
print(f"  building fraction: {building_pct}%  (→ higher = more self-shading already)")

In [ ]:
import base64, io
from PIL import Image

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, title in [(axes[0], 'original_image',  f"Street view at {hot1['stop_id']}"),
                       (axes[1], 'segmented_image', 'Segmentation')]:
    img = _decode(front.get(key))
    if img is not None: ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

---
## Step 8 — Environmental drivers through the day

### What you are doing
Profiling environmental parameters at the #1 stop from 07:00 to 19:00 on the study date. We plot heat index and relative humidity through the day so you can see *when* discomfort peaks, not just how hot it gets at one instant.

### Why this matters
A stop that is unbearable from 12:00–17:00 demands different intervention timing than one that peaks during evening commute. The hour-by-hour profile tells you whether shade, misting, or ventilation is the right fix — and whether the fix needs to be passive (works all day) or active (runs only during peak).

In [ ]:
env = client.environmental_parameters(
    latitude=hot1.latitude, longitude=hot1.longitude,
    temperature=float(hot1.temperature_c),
    start_date=STUDY_DATE, start_time='07:00', end_time='19:00',
    filter_type=2, verbose=False,
)
res    = env['result']
loc    = res['locations'][0]
params = loc.get('parameters', {})
ts     = pd.to_datetime(res['metadata'].get('timestamps', []))

env_df = pd.DataFrame({k: v for k, v in params.items()
                       if isinstance(v, list) and len(v) == len(ts)})
env_df.insert(0, 'timestamp', ts); env_df.set_index('timestamp', inplace=True)

plot_cols = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                         'wet_bulb_temperature_celsius', 'relative_humidity_percent')
             if c in env_df.columns]
if plot_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    thermal = [c for c in plot_cols if 'humidity' not in c]
    if thermal: env_df[thermal].plot(ax=axes[0], marker='o'); axes[0].set_title(f"Thermal comfort at {hot1['stop_id']}"); axes[0].set_ylabel('°C'); axes[0].grid(alpha=0.3)
    if 'relative_humidity_percent' in env_df.columns:
        env_df['relative_humidity_percent'].plot(ax=axes[1], color='steelblue', marker='o')
        axes[1].set_title('Relative humidity'); axes[1].set_ylabel('%'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

peak_hi = env_df['heat_index_celsius'].max() if 'heat_index_celsius' in env_df.columns else None
peak_hour = env_df['heat_index_celsius'].idxmax() if 'heat_index_celsius' in env_df.columns else None
print(f"Peak heat index: {peak_hi}  at {peak_hour}")

---
## Step 9 — Prioritized action list

### What you are doing
Combining everything we learned into a single action-oriented DataFrame. For each of the top stops, a simple heuristic picks the **dominant driver** and maps it to a **recommended intervention**. The rules are:

- `vegetation_pct < 10%` → **Plant street trees** (shade + transpiration cooling)
- `impervious_pct > 70%` → **Cool surfaces / reflective paving** (reduce absorbed radiation)
- `sky_pct > 45%` at the #1 stop → **Install shade structure** (most of the sky is open overhead)
- `peak heat index > 32 °C` → **Add active cooling** (misting or passive shelter ventilation)

Multiple rules can fire for one stop — we keep the strongest one plus the full rationale.

### Why this matters
This is the row the council actually reads. Every column is traceable:
- `temperature_c` — comes from step 3
- `impervious_pct` / `vegetation_pct` — step 6
- `recommended_action` — derived here from steps 6, 7, 8

You can defend every recommendation with the diagnostic chain behind it.

In [ ]:
def _recommend(row, sky_pct_row1: float | None, peak_hi: float | None, is_top1: bool):
    reasons, actions = [], []
    veg = row.get('vegetation_pct')
    imp = row.get('impervious_pct')
    if pd.notna(veg) and veg < 10:
        reasons.append(f'vegetation {veg}% (low)')
        actions.append('Plant street trees')
    if pd.notna(imp) and imp > 70:
        reasons.append(f'impervious {imp}% (high)')
        actions.append('Cool surfaces / reflective paving')
    if is_top1 and sky_pct_row1 is not None and sky_pct_row1 > 45:
        reasons.append(f'sky exposure {sky_pct_row1}% (open)')
        actions.append('Install shade structure')
    if peak_hi is not None and peak_hi > 32:
        reasons.append(f'peak heat index {peak_hi:.1f} °C')
        actions.append('Add active cooling (misting)')
    return pd.Series({
        'dominant_driver': reasons[0] if reasons else 'temperature only',
        'recommended_action': ' + '.join(dict.fromkeys(actions)) or 'Monitor',
        'rationale': '; '.join(reasons) or f"tile temperature {row.get('temperature_c')} °C",
    })

actions = top.apply(
    lambda r: _recommend(r, sky_pct if r['rank'] == 1 else None, peak_hi, r['rank'] == 1),
    axis=1,
)
action_list = pd.concat([top.reset_index(drop=True),
                          actions.reset_index(drop=True)], axis=1)

display_cols = ['rank', 'stop_id', 'name', 'temperature_c',
                'impervious_pct', 'vegetation_pct',
                'dominant_driver', 'recommended_action']
action_list[display_cols]

In [ ]:
# Export the action list to hand off to the operations team.
out_path = ROOT / 'outputs' / 'bus_stop_action_list.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
action_list.to_csv(out_path, index=False)
print(f'Saved prioritized action list to {out_path}')

---
## Wrap-up — what you now have

Starting from a single CSV of bus stops you now have:

| Artifact | Step | Audience |
|----------|------|----------|
| Temperature-joined stops table | 3 | GIS / analytics team |
| Visual hotspot ranking on the city map | 4 | Council presentation |
| Above-mean hotspot cluster map | 5 | Council presentation |
| Macro diagnosis of the top-N (impervious / vegetation %) | 6 | Landscape / infrastructure team |
| Street-level confirmation of the #1 stop | 7 | Design review |
| Diurnal heat-index profile at the worst location | 8 | Intervention-type selection |
| Prioritized action list CSV | 9 | Operations hand-off |

Every action in the final list is traceable back to a measurement — not an assumption. That is the defensibility the council was missing, and the reason this workflow scales beyond bus stops.

### Apply this pattern to your other layers

The workflow is agnostic to what kind of asset your points represent. Swap the CSV for any point layer and everything downstream works:

- **Schools / playgrounds** → prioritize which outdoor spaces need tree planting
- **Public benches / transit shelters** → identify which need upgrading to reflective / vented designs
- **Bike-share docks** → identify stations where riders drop off because they overheat
- **Utility substations / pumping stations** → identify infrastructure at heat-failure risk
- **Social-housing units** → identify buildings in hottest blocks for retrofit prioritization

The pattern — **your geometries × our thermal, surface, and environmental layers → ranked actions** — is the whole point.